In [75]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
import numpy as np


In [76]:
try:
    churn_df = pd.read_csv('churn.txt')
    print("Successfully read churn.txt")
except FileNotFoundError:
    print("Error: churn.txt not found. Please make sure the file is in the correct directory.")
    exit()
print(churn_df.head())

Successfully read churn.txt
  State  Account Length  Area Code     Phone Int'l Plan VMail Plan  \
0    KS             128        415  382-4657         no        yes   
1    OH             107        415  371-7191         no        yes   
2    NJ             137        415  358-1921         no         no   
3    OH              84        408  375-9999        yes         no   
4    OK              75        415  330-6626        yes         no   

   VMail Message  Day Mins  Day Calls  Day Charge  ...  Eve Calls  Eve Charge  \
0             25     265.1        110       45.07  ...         99       16.78   
1             26     161.6        123       27.47  ...        103       16.62   
2              0     243.4        114       41.38  ...        110       10.30   
3              0     299.4         71       50.90  ...         88        5.26   
4              0     166.7        113       28.34  ...        122       12.61   

   Night Mins  Night Calls  Night Charge  Intl Mins  Intl Calls 

In [77]:
try:
    dict_df = pd.read_csv('dictchurn.txt', header=None, names=['NewValue', 'OriginalValue_yn', 'OriginalValue_tf'])
    print("Successfully read dictchurn.txt")
    print(dict_df)
except FileNotFoundError:
    print("Error: dictchurn.txt not found. Please make sure the file is in the correct directory.")
    exit()


Successfully read dictchurn.txt
   NewValue OriginalValue_yn OriginalValue_tf
0         0               no           False.
1         1              yes            True.


In [78]:
columns_to_drop = ['Phone', 'State']
churn_df_filtered = churn_df.drop(columns=columns_to_drop)
print(f"Dropped columns: {columns_to_drop}")

Dropped columns: ['Phone', 'State']


In [81]:
yes_no_dict = dict_df.set_index('OriginalValue_yn')['NewValue'].to_dict()
print("\nyes_no_dict mapping:", yes_no_dict)
def replace_values(df, column, replacement_dict):
    if column in df.columns:
        df[column] = df[column].map(replacement_dict).fillna(df[column])
    return df

churn_df_replaced_yn = churn_df_filtered.copy()
for col in ['Int\'l Plan', 'VMail Plan']:
    churn_df_replaced_yn = replace_values(churn_df_replaced_yn, col, yes_no_dict)
print("Applied replacements for 'Int\'l Plan' and 'Vmail Plan'.")


yes_no_dict mapping: {'no': 0, 'yes': 1}
Applied replacements for 'Int'l Plan' and 'Vmail Plan'.


In [82]:
print(churn_df_replaced_yn.head())
print(churn_df_replaced_yn["VMail Plan"].unique())

   Account Length  Area Code  Int'l Plan  VMail Plan  VMail Message  Day Mins  \
0             128        415           0           1             25     265.1   
1             107        415           0           1             26     161.6   
2             137        415           0           0              0     243.4   
3              84        408           1           0              0     299.4   
4              75        415           1           0              0     166.7   

   Day Calls  Day Charge  Eve Mins  Eve Calls  Eve Charge  Night Mins  \
0        110       45.07     197.4         99       16.78       244.7   
1        123       27.47     195.5        103       16.62       254.4   
2        114       41.38     121.2        110       10.30       162.6   
3         71       50.90      61.9         88        5.26       196.9   
4        113       28.34     148.3        122       12.61       186.9   

   Night Calls  Night Charge  Intl Mins  Intl Calls  Intl Charge  \
0     

In [41]:
true_false_dict = dict_df.set_index('OriginalValue_tf')['NewValue'].to_dict()
churn_df_replaced_tf = replace_values(churn_df_replaced_yn.copy(), 'Churn?', true_false_dict)
print("Applied replacements for 'Churn?'.")


Applied replacements for 'Churn?'.


In [42]:
columns_to_numeric = ['Int\'l Plan', 'Vmail Plan', 'Churn?']
churn_df_numeric = churn_df_replaced_tf.copy()
for col in columns_to_numeric:
    if col in churn_df_numeric.columns:
        churn_df_numeric[col] = pd.to_numeric(churn_df_numeric[col], errors='coerce')
        print(f"Converted column '{col}' to numeric.")
    else:
        print(f"Warning: Column '{col}' not found for numeric conversion.")


Converted column 'Int'l Plan' to numeric.
Converted column 'Churn?' to numeric.


In [43]:
non_numeric_cols = churn_df_numeric.select_dtypes(include=['object']).columns
print(f"\nRemaining non-numeric columns before clustering: {list(non_numeric_cols)}")



Remaining non-numeric columns before clustering: ['VMail Plan']


In [44]:
label_encoders = {}
for column in non_numeric_cols:
    label_encoders[column] = LabelEncoder()
    churn_df_numeric[column] = label_encoders[column].fit_transform(churn_df_numeric[column])
print("Encoded remaining categorical columns.")


Encoded remaining categorical columns.


In [45]:
X = churn_df_numeric.drop(columns=['Churn?'], errors='ignore')
print("\nPrepared data for clustering (excluding 'Churn?').")
print(X.head())


Prepared data for clustering (excluding 'Churn?').
   Account Length  Area Code  Int'l Plan  VMail Plan  VMail Message  Day Mins  \
0             128        415           0           1             25     265.1   
1             107        415           0           1             26     161.6   
2             137        415           0           0              0     243.4   
3              84        408           1           0              0     299.4   
4              75        415           1           0              0     166.7   

   Day Calls  Day Charge  Eve Mins  Eve Calls  Eve Charge  Night Mins  \
0        110       45.07     197.4         99       16.78       244.7   
1        123       27.47     195.5        103       16.62       254.4   
2        114       41.38     121.2        110       10.30       162.6   
3         71       50.90      61.9         88        5.26       196.9   
4        113       28.34     148.3        122       12.61       186.9   

   Night Calls  Night 

In [46]:
n_components_em = 3
em = GaussianMixture(n_components=n_components_em, random_state=42, init_params='kmeans')
em_labels = em.fit_predict(X)
print(f"Applied Gaussian Mixture Model with {n_components_em} components.")

Applied Gaussian Mixture Model with 3 components.


In [47]:
clustered_data_em = X.copy()
clustered_data_em['EM_Cluster'] = em_labels
print("\nClustered data using E-M (first few rows):")
print(clustered_data_em.head())


Clustered data using E-M (first few rows):
   Account Length  Area Code  Int'l Plan  VMail Plan  VMail Message  Day Mins  \
0             128        415           0           1             25     265.1   
1             107        415           0           1             26     161.6   
2             137        415           0           0              0     243.4   
3              84        408           1           0              0     299.4   
4              75        415           1           0              0     166.7   

   Day Calls  Day Charge  Eve Mins  Eve Calls  Eve Charge  Night Mins  \
0        110       45.07     197.4         99       16.78       244.7   
1        123       27.47     195.5        103       16.62       254.4   
2        114       41.38     121.2        110       10.30       162.6   
3         71       50.90      61.9         88        5.26       196.9   
4        113       28.34     148.3        122       12.61       186.9   

   Night Calls  Night Charge  

In [48]:
print("\nDescriptive statistics of clusters formed by E-M:")
print(clustered_data_em.groupby('EM_Cluster').mean())



Descriptive statistics of clusters formed by E-M:
            Account Length   Area Code  Int'l Plan  VMail Plan  VMail Message  \
EM_Cluster                                                                      
0               100.907570  412.636884    0.000000    0.274208       8.028169   
1               105.791855  412.751131    1.000000    0.307692       9.131222   
2               100.246429  510.000000    0.121429    0.275000       8.019048   

              Day Mins   Day Calls  Day Charge    Eve Mins   Eve Calls  \
EM_Cluster                                                               
0           179.166725  100.462148   30.458886  200.690581  100.185299   
1           189.782805  101.447964   32.263620  202.653394  101.067873   
2           178.787619  100.097619   30.394429  201.323929   99.671429   

            Eve Charge  Night Mins  Night Calls  Night Charge  Intl Mins  \
EM_Cluster                                                                 
0            17.0588

In [49]:
kmeans = KMeans(n_clusters=n_components_em, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X)
print(f"Applied K-means with {n_components_em} clusters.")


Applied K-means with 3 clusters.


In [50]:
clustered_data_kmeans = X.copy()
clustered_data_kmeans['KMeans_Cluster'] = kmeans_labels
print("\nClustered data using K-means (first few rows):")
print(clustered_data_kmeans.head())


Clustered data using K-means (first few rows):
   Account Length  Area Code  Int'l Plan  VMail Plan  VMail Message  Day Mins  \
0             128        415           0           1             25     265.1   
1             107        415           0           1             26     161.6   
2             137        415           0           0              0     243.4   
3              84        408           1           0              0     299.4   
4              75        415           1           0              0     166.7   

   Day Calls  Day Charge  Eve Mins  Eve Calls  Eve Charge  Night Mins  \
0        110       45.07     197.4         99       16.78       244.7   
1        123       27.47     195.5        103       16.62       254.4   
2        114       41.38     121.2        110       10.30       162.6   
3         71       50.90      61.9         88        5.26       196.9   
4        113       28.34     148.3        122       12.61       186.9   

   Night Calls  Night Char

In [52]:
print("\nDescriptive statistics of clusters formed by K-means:")
print(clustered_data_kmeans.groupby('KMeans_Cluster').mean())

print("\nDistribution of 'Int\'l Plan' and 'Vmail Plan' in K-means clusters:")
print(clustered_data_kmeans.groupby('KMeans_Cluster')[['Int\'l Plan', 'VMail Plan']].mean())



Descriptive statistics of clusters formed by K-means:
                Account Length   Area Code  Int'l Plan  VMail Plan  \
KMeans_Cluster                                                       
0                    99.998783  510.000000    0.121655    0.274939   
1                   102.730994  413.944862    0.100251    0.289891   
2                   100.213851  412.798326    0.078387    0.265601   

                VMail Message    Day Mins   Day Calls  Day Charge    Eve Mins  \
KMeans_Cluster                                                                  
0                    8.051095  177.015207  100.120438   30.093127  201.202068   
1                    8.537176  225.494486  100.309106   38.334570  205.647953   
2                    7.729833  139.853120  100.748097   23.775601  196.589650   

                 Eve Calls  Eve Charge  Night Mins  Night Calls  Night Charge  \
KMeans_Cluster                                                                  
0                99.509732

In [53]:
scaler = MinMaxScaler()
X_normalized = scaler.fit_transform(X)
X_normalized_df = pd.DataFrame(X_normalized, columns=X.columns)
print("\nData normalized using Min-Max scaling (first few rows):")
print(X_normalized_df.head())


Data normalized using Min-Max scaling (first few rows):
   Account Length  Area Code  Int'l Plan  VMail Plan  VMail Message  Day Mins  \
0        0.524793   0.068627         0.0         1.0       0.490196  0.755701   
1        0.438017   0.068627         0.0         1.0       0.509804  0.460661   
2        0.561983   0.068627         0.0         0.0       0.000000  0.693843   
3        0.342975   0.000000         1.0         0.0       0.000000  0.853478   
4        0.305785   0.068627         1.0         0.0       0.000000  0.475200   

   Day Calls  Day Charge  Eve Mins  Eve Calls  Eve Charge  Night Mins  \
0   0.666667    0.755701  0.542755   0.582353    0.542866    0.595750   
1   0.745455    0.460597  0.537531   0.605882    0.537690    0.621840   
2   0.690909    0.693830  0.333242   0.647059    0.333225    0.374933   
3   0.430303    0.853454  0.170195   0.517647    0.170171    0.467187   
4   0.684848    0.475184  0.407754   0.717647    0.407959    0.440290   

   Night Calls  N

In [54]:
kmeans_normalized = KMeans(n_clusters=n_components_em, random_state=42, n_init=10)
kmeans_normalized_labels = kmeans_normalized.fit_predict(X_normalized)

clustered_data_kmeans_normalized = X_normalized_df.copy()
clustered_data_kmeans_normalized['KMeans_Normalized_Cluster'] = kmeans_normalized_labels
print("\nClustered data using K-means on normalized data (first few rows):")
print(clustered_data_kmeans_normalized.head())

print("\nDescriptive statistics of clusters formed by K-means on normalized data:")
print(clustered_data_kmeans_normalized.groupby('KMeans_Normalized_Cluster').mean())




Clustered data using K-means on normalized data (first few rows):
   Account Length  Area Code  Int'l Plan  VMail Plan  VMail Message  Day Mins  \
0        0.524793   0.068627         0.0         1.0       0.490196  0.755701   
1        0.438017   0.068627         0.0         1.0       0.509804  0.460661   
2        0.561983   0.068627         0.0         0.0       0.000000  0.693843   
3        0.342975   0.000000         1.0         0.0       0.000000  0.853478   
4        0.305785   0.068627         1.0         0.0       0.000000  0.475200   

   Day Calls  Day Charge  Eve Mins  Eve Calls  Eve Charge  Night Mins  \
0   0.666667    0.755701  0.542755   0.582353    0.542866    0.595750   
1   0.745455    0.460597  0.537531   0.605882    0.537690    0.621840   
2   0.690909    0.693830  0.333242   0.647059    0.333225    0.374933   
3   0.430303    0.853454  0.170195   0.517647    0.170171    0.467187   
4   0.684848    0.475184  0.407754   0.717647    0.407959    0.440290   

   Nigh

In [56]:
# 8. Compare results
print("\nDistribution of normalized 'Int\'l Plan' and 'Vmail Plan' in K-means clusters (normalized data):")
print(clustered_data_kmeans_normalized.groupby('KMeans_Normalized_Cluster')[['Int\'l Plan', 'VMail Plan']].mean())

print("\nComparison of cluster characteristics (mean values) might reveal how normalization affects the segmentation, especially for features with different scales.")
print("For example, features with larger original ranges might have disproportionately influenced the clustering before normalization.")
print("After normalization, all features are on the same scale (0-1), potentially leading to clusters that are more influenced by the patterns across all attributes equally.")
print("Observe how the mean values of 'Int\'l Plan' and 'Vmail Plan' differ across clusters before and after normalization.")


Distribution of normalized 'Int'l Plan' and 'Vmail Plan' in K-means clusters (normalized data):
                           Int'l Plan  VMail Plan
KMeans_Normalized_Cluster                        
0                            0.084906         0.0
1                            0.099783         1.0
2                            0.128079         0.0

Comparison of cluster characteristics (mean values) might reveal how normalization affects the segmentation, especially for features with different scales.
For example, features with larger original ranges might have disproportionately influenced the clustering before normalization.
After normalization, all features are on the same scale (0-1), potentially leading to clusters that are more influenced by the patterns across all attributes equally.
Observe how the mean values of 'Int'l Plan' and 'Vmail Plan' differ across clusters before and after normalization.
